# Avaliacao RAGAS - Qualidade do RAG com Ground Truth

Este notebook usa a biblioteca **RAGAS** para avaliar a qualidade do pipeline RAG
contra um **ground truth** (respostas esperadas escritas por nos).

Avaliamos 4 metricas:

- **faithfulness**: a resposta esta fundamentada nos chunks recuperados? (nao alucina?)
- **answer_relevancy**: a resposta responde de fato a pergunta?
- **context_precision**: os chunks recuperados sao relevantes (vem no topo)?
- **context_recall**: os chunks recuperados cobrem o que o ground truth exige?

As duas ultimas (`context_precision` e `context_recall`) sao as que comparam
contra o ground truth - exatamente o que o RT pediu.

## 0. Setup

**Pre-requisitos** (rode no terminal, com o .venv ativo):

```
pip install ragas==0.1.21 datasets langchain-openai nest_asyncio
```

E o ChromaDB precisa estar populado: `python -m src.rag.ingestion`.

> Voce precisa preencher o nome do seu **deployment de chat** da Azure na
> constante `CHAT_DEPLOYMENT` abaixo (o mesmo que o `AzureModel` usa).

In [4]:
import os
import sys
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()  # RAGAS usa asyncio; isso evita conflito dentro do Jupyter

# O notebook esta em notebooks/ -> sobe um nivel para a raiz do projeto
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))

from src.api.schemas import AnalysisRequest
from src.core.llm_client import AzureModel
from src.services.complience_service import analyze_recommendation, fused_retrieval

# >>> PREENCHA com o nome do seu deployment de chat da Azure <<<
CHAT_DEPLOYMENT = "gpt-4o-mini"
EMBEDDING_DEPLOYMENT = "text-embedding-ada-002"

print(f"Working directory: {Path.cwd()}")
print("Setup concluido.")

Working directory: c:\Users\BB442HD\OneDrive - EY\Desktop\compliance-viewer
Setup concluido.


## 1. Casos de teste com ground truth

Cada caso tem:
- `text`: a recomendacao de investimento
- `client_profile`: o perfil do cliente
- `ground_truth`: a resposta esperada (o gabarito que NOS escrevemos)

Ajuste/adicione casos conforme a sua knowledge base. Quanto mais o ground_truth
refletir o que esta nos documentos, mais justa a avaliacao.

In [5]:
test_cases = [
    {
        "text": "Recomendo alocar 80% da carteira em acoes small cap e criptomoedas.",
        "client_profile": "conservador",
        "ground_truth": (
            "Recomendacao nao conforme: produtos de alto risco como acoes small cap "
            "e criptomoedas sao inadequados para um perfil conservador, que exige "
            "predominancia de renda fixa e baixa tolerancia a volatilidade."
        ),
    },
    {
        "text": "Sugiro um CDB de banco grande e Tesouro Selic para a reserva.",
        "client_profile": "conservador",
        "ground_truth": (
            "Recomendacao conforme: CDB e Tesouro Selic sao produtos de baixo risco "
            "e boa liquidez, adequados ao perfil conservador."
        ),
    },
    {
        "text": "Indico um fundo multimercado com exposicao moderada a renda variavel.",
        "client_profile": "moderado",
        "ground_truth": (
            "Recomendacao conforme: fundo multimercado com exposicao moderada e "
            "compativel com o perfil moderado, que tolera risco intermediario."
        ),
    },
]

print(f"{len(test_cases)} casos de teste definidos.")

3 casos de teste definidos.


## 2. Rodando o pipeline e montando o dataset

Para cada caso:
1. `fused_retrieval` -> pega os chunks usados como **contexto**
2. `analyze_recommendation` -> gera a **resposta** (campo `reason`)
3. Juntamos pergunta + resposta + contexto + ground_truth

> Observacao honesta: o RAG Fusion reescreve a query a cada chamada, entao o
> contexto pode variar um pouco entre execucoes. Para uma avaliacao estavel,
> rode o notebook inteiro de uma vez.

In [6]:
from datasets import Dataset
import time

llm_client = AzureModel()

questions, answers, contexts_list, ground_truths = [], [], [], []

for case in test_cases:
    req = AnalysisRequest(text=case["text"], client_profile=case["client_profile"])

    # Contexto: os chunks recuperados pelo RAG Fusion
    chunks = fused_retrieval(case["text"], llm_client)
    contexts = [c["text"] for c in chunks]

    # Resposta gerada pelo pipeline completo
    result = analyze_recommendation(req)

    questions.append(f"[{case['client_profile']}] {case['text']}")
    answers.append(result.reason)
    contexts_list.append(contexts)
    ground_truths.append(case["ground_truth"])

    print(f"OK: {case['text'][:50]}... | compliant={result.is_compliant}")
    time.sleep(20)

dataset = Dataset.from_dict({
    "question": questions,
    "answer": answers,
    "contexts": contexts_list,
    "ground_truth": ground_truths,
})

print(f"\nDataset montado com {len(dataset)} amostras.")

OK: Recomendo alocar 80% da carteira em acoes small ca... | compliant=False
OK: Sugiro um CDB de banco grande e Tesouro Selic para... | compliant=True
OK: Indico um fundo multimercado com exposicao moderad... | compliant=True

Dataset montado com 3 amostras.


## 3. Avaliando com RAGAS

Configuramos o RAGAS para usar o **seu** Azure OpenAI como juiz (LLM) e como
gerador de embeddings. As 4 metricas rodam sobre o dataset montado acima.

In [7]:
from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)
from langchain_openai import AzureChatOpenAI, AzureOpenAIEmbeddings

judge_llm = AzureChatOpenAI(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=CHAT_DEPLOYMENT,
)

judge_emb = AzureOpenAIEmbeddings(
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_key=os.getenv("AZURE_OPENAI_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION"),
    azure_deployment=EMBEDDING_DEPLOYMENT,
)

scores = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
    llm=judge_llm,
    embeddings=judge_emb,
)

print(scores)

Evaluating: 100%|██████████| 12/12 [02:23<00:00, 11.98s/it]


{'faithfulness': 0.2222, 'answer_relevancy': 0.8574, 'context_precision': 0.8333, 'context_recall': 1.0000}


## 4. Resultado detalhado e conclusao

A tabela abaixo mostra a nota de cada metrica por caso. Valores vao de 0 a 1
(mais perto de 1 = melhor).

In [8]:
df = scores.to_pandas()
df

,question,answer,contexts,ground_truth,faithfulness,answer_relevancy,context_precision,context_recall
0,[conservador] Recomendo alocar 80% da carteira...,A recomendação de alocar 80% da carteira em aç...,"[""Olá, Sr. João!\n\nAnalisando seu portfólio, ...",Recomendacao nao conforme: produtos de alto ri...,0.333333,0.857167,0.5,1.0
1,[conservador] Sugiro um CDB de banco grande e ...,O Tesouro Selic e CDBs de bancos grandes são c...,[**3. Classificação de Risco dos Produtos**\n-...,Recomendacao conforme: CDB e Tesouro Selic sao...,0.333333,0.798179,1.0,1.0
2,[moderado] Indico um fundo multimercado com ex...,Fundos Multimercado com exposição moderada a r...,[**3. Classificação de Risco dos Produtos**\n-...,Recomendacao conforme: fundo multimercado com ...,0.000000,0.916961,1.0,1.0


### Como ler (para a defesa)

- **faithfulness alto**: o agente nao inventa - tudo que ele afirma vem dos chunks.
- **answer_relevancy alto**: a resposta foi direto ao ponto da pergunta.
- **context_precision alto**: o retrieval + re-ranking colocaram o que importa no topo.
- **context_recall alto**: o retrieval trouxe tudo que o ground_truth precisava.

Se `context_recall` vier baixo em algum caso, significa que faltou documento
relevante na knowledge base (ou o chunking cortou a informacao). Esse e o tipo
de insight que o RAGAS entrega e que justifica a avaliacao.